In [0]:
from pyspark.sql import functions as F

S = "workspace.olist"
fact = spark.table(f"{S}.silver_order_fact")
delivered = fact.filter("order_status = 'delivered'")

# 1. Monthly revenue by category
monthly = (delivered
    .withColumn("month", F.to_date(F.date_trunc("month", "order_purchase_timestamp")))
    .groupBy("month", "category_en")
    .agg(F.round(F.sum("price"), 2).alias("revenue"),
         F.round(F.sum("freight_value"), 2).alias("freight"),
         F.countDistinct("order_id").alias("orders"),
         F.count("*").alias("items")))
monthly.write.mode("overwrite").format("delta").saveAsTable(f"{S}.gold_monthly_revenue_by_category")

# 2. Delivery performance by customer state (order level)
order_level = (delivered
    .select("order_id", "customer_state", "is_late", "delivery_days", "estimated_days", "avg_review_score")
    .dropDuplicates(["order_id"]))
by_state = (order_level.groupBy("customer_state")
    .agg(F.count("*").alias("orders"),
         F.round(F.avg("is_late") * 100, 2).alias("late_pct"),
         F.round(F.avg("delivery_days"), 1).alias("avg_delivery_days"),
         F.round(F.avg("estimated_days"), 1).alias("avg_estimated_days"),
         F.round(F.avg("avg_review_score"), 2).alias("avg_review")))
by_state.write.mode("overwrite").format("delta").saveAsTable(f"{S}.gold_delivery_by_state")

# 3. Seller performance
seller_order = (delivered.groupBy("seller_id", "seller_state", "order_id")
    .agg(F.sum("price").alias("rev"), F.max("is_late").alias("is_late"),
         F.avg("avg_review_score").alias("review")))
sellers_g = (seller_order.groupBy("seller_id", "seller_state")
    .agg(F.count("*").alias("orders"),
         F.round(F.sum("rev"), 2).alias("revenue"),
         F.round(F.avg("is_late") * 100, 2).alias("late_pct"),
         F.round(F.avg("review"), 2).alias("avg_review")))
sellers_g.write.mode("overwrite").format("delta").saveAsTable(f"{S}.gold_seller_performance")

# 4. Review score vs lateness
review_late = (order_level.groupBy("is_late")
    .agg(F.count("*").alias("orders"), F.round(F.avg("avg_review_score"), 2).alias("avg_review")))
review_late.write.mode("overwrite").format("delta").saveAsTable(f"{S}.gold_review_vs_lateness")

# 5. Customer metrics
cust_order = (fact.groupBy("customer_unique_id", "order_id")
    .agg(F.sum("price").alias("order_value"), F.min("order_purchase_timestamp").alias("ts")))
cust = (cust_order.groupBy("customer_unique_id")
    .agg(F.count("*").alias("orders"), F.round(F.sum("order_value"), 2).alias("total_spend"),
         F.min("ts").alias("first_order"), F.max("ts").alias("last_order"))
    .withColumn("is_repeat", (F.col("orders") > 1).cast("int")))
cust.write.mode("overwrite").format("delta").saveAsTable(f"{S}.gold_customer_metrics")

for t in ["gold_monthly_revenue_by_category", "gold_delivery_by_state", "gold_seller_performance",
          "gold_review_vs_lateness", "gold_customer_metrics"]:
    print(t, spark.table(f"{S}.{t}").count())

gold_monthly_revenue_by_category 1273
gold_delivery_by_state 27
gold_seller_performance 2970
gold_review_vs_lateness 3
gold_customer_metrics 95420


In [0]:
# Chart 1: monthly revenue trend (line)
display(spark.table(f"{S}.gold_monthly_revenue_by_category")
        .groupBy("month").agg(F.sum("revenue").alias("revenue")).orderBy("month"))

month,revenue
2016-09-01,134.97
2016-10-01,40325.10999999999
2016-12-01,10.9
2017-01-01,111798.36
2017-02-01,234223.39999999997
2017-03-01,359198.85000000003
2017-04-01,340669.68000000017
2017-05-01,489338.25
2017-06-01,421923.3700000001
2017-07-01,481604.51999999996


Databricks visualization. Run in Databricks to view.

In [0]:
# Chart 2: late delivery rate by state (bar)
display(spark.table(f"{S}.gold_delivery_by_state").orderBy(F.desc("late_pct")))

customer_state,orders,late_pct,avg_delivery_days,avg_estimated_days,avg_review
AL,397,23.93,24.5,33.2,3.85
MA,717,19.67,21.5,31.1,3.83
PI,476,15.97,19.4,30.7,3.99
CE,1279,15.32,21.2,32.0,3.94
SE,335,15.22,21.5,31.5,3.91
BA,3256,14.04,19.3,30.1,3.93
RJ,12350,13.47,15.2,27.0,3.97
TO,274,12.77,17.6,29.7,4.15
PA,946,12.37,23.7,37.8,3.91
ES,1995,12.23,15.7,26.2,4.08


Databricks visualization. Run in Databricks to view.

In [0]:
# Chart 3: top 10 categories by revenue (bar)
display(spark.table(f"{S}.gold_monthly_revenue_by_category")
        .groupBy("category_en").agg(F.round(F.sum("revenue"), 0).alias("revenue"))
        .orderBy(F.desc("revenue")).limit(10))

category_en,revenue
health_beauty,1233132.0
watches_gifts,1166177.0
bed_bath_table,1023435.0
sports_leisure,954853.0
computers_accessories,888725.0
furniture_decor,711928.0
housewares,615629.0
cool_stuff,610204.0
auto,578967.0
toys,471286.0


Databricks visualization. Run in Databricks to view.

In [0]:
display(spark.table(f"{S}.gold_review_vs_lateness")
        .filter("is_late IS NOT NULL")
        .withColumn("status", F.when(F.col("is_late") == 1, "Late").otherwise("On time"))
        .select("status", "orders", "avg_review"))

status,orders,avg_review
Late,7826,2.57
On time,88644,4.29


Databricks visualization. Run in Databricks to view.

In [0]:
c = spark.table(f"{S}.gold_customer_metrics")
c.agg(F.count("*").alias("customers"), F.round(F.avg("is_repeat") * 100, 2).alias("repeat_pct")).show()

+---------+----------+
|customers|repeat_pct|
+---------+----------+
|    95420|      3.05|
+---------+----------+

